In [1]:
import polars as pl
import sqlite3
import numpy as np
import pandas as pd
from tqdm import tqdm
from sqlitedict import SqliteDict
from collections import defaultdict
# import swifter

# from bam_to_sqlite_anmol import nuc_count, dinuc_count, table2dict


In [2]:
pl.__version__

!rm anmol.sql
dct = SqliteDict("anmol.sql",autocommit=True,)
dct["description"] = 'NULL'
dct["desc"] = 'NULL'
dct["cutadapt_removed"] =0 


In [3]:
# Things to find

[#'unmapped_reads',
 'unambiguous_threeprime_totals',
 'unambiguous_fiveprime_totals',
 'unambiguous_cds_totals',
 'unambiguous_all_totals',
 #'unambig_read_lengths',
 'trip_periodicity',
 # 'threeprime_nuc_counts', # Dunno use of it
 'stop_metagene_counts',
 # 'rrna_removed', # For later. it is not in server code
 'removed_minus_m',
 #'read_lengths',
 'offsets',
 #'nuc_counts',
 #'noncoding_counts',
 'metagene_counts',
 #'mapped_reads',
 #'frequent_unmapped_reads',
 # 'dinuc_counts',
 #'description',
 #'desc',
 #'cutadapt_removed',
 #'coding_counts',
 'ambiguous_threeprime_totals',
 'ambiguous_fiveprime_totals',
 #'ambiguous_counts',
 'ambiguous_cds_totals',
 'ambiguous_all_totals',
    ]

['unambiguous_threeprime_totals',
 'unambiguous_fiveprime_totals',
 'unambiguous_cds_totals',
 'unambiguous_all_totals',
 'trip_periodicity',
 'stop_metagene_counts',
 'removed_minus_m',
 'offsets',
 'metagene_counts',
 'ambiguous_threeprime_totals',
 'ambiguous_fiveprime_totals',
 'ambiguous_cds_totals',
 'ambiguous_all_totals']

In [4]:
sqlite_file = "./homo_sapiens.Gencode_v25.sqlite"
conn = sqlite3.connect(sqlite_file)

In [5]:
%%time 

samfile = pd.read_csv("./Galaxy93-[Bowtie_Transcriptome_Alignment_on_data_92__mapped_reads].sam",sep="\t", comment="@", header=None, names=range(14), low_memory=True).drop([1,4,5,6,
 7,8,10,11], axis='columns').rename(columns = {0:"qname",2:"rname",3:"pos",9:"seq",12:"MD",13:"NM"}) #,nrows=500000
samfile["transcript"] = samfile.rname.apply(lambda x: x.split("|")[0].split(".")[0])
# samfile = samfile[samfile["transcript"]=="ENST00000638157"]


# Read length distribution
read_lens= defaultdict( int )
for w in samfile.drop_duplicates(["qname","seq"])["seq"].apply(len):
    read_lens[w] += 1
    
dct["read_lengths"] = dict(read_lens)




samfile_unmapped = samfile.loc[samfile["rname"]=="*",["seq"]]
#unmapped reads
dct["samfile_unmapped"] = samfile_unmapped.shape[0]




samfile_unmapped = samfile_unmapped.groupby('seq').size().reset_index().rename(columns={0:'len'})
samfile_mapped = samfile[samfile["rname"]!="*"]
samfile_mapped["transcript"] = samfile_mapped["rname"].apply(lambda x:x.split(".")[0])
del samfile_mapped["rname"]
print("Alignments in bam file")
print(samfile.shape)
print("Reads in Bam File")
print(samfile.drop_duplicates("qname").shape)

del samfile



<timed exec>:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Alignments in bam file
(11791325, 7)
Reads in Bam File
(6953524, 7)
CPU times: user 1min, sys: 9.71 s, total: 1min 10s
Wall time: 1min 14s


In [6]:
print("UnMapped,Mapped")
sum(samfile_unmapped["len"]), samfile_mapped.drop_duplicates("qname").shape

UnMapped,Mapped


(6327376, (626148, 6))

In [7]:
transcriptome = pl.read_database(
    query="SELECT * FROM transcripts", 
    connection=conn,
).filter(pl.col("transcript").is_in(list(set(samfile_mapped["transcript"])))).select(["transcript","cds_start","cds_stop","length","strand","chrom","tran_type"])#.filter(pl.col("tran_type")==1)

In [9]:
set(samfile_mapped["transcript"]) - set(transcriptome["transcript"])

set()

In [8]:
exons = pl.read_database(
    query="SELECT * FROM exons", 
    connection=conn,
).filter(pl.col("transcript").is_in(set(samfile_mapped["transcript"]))).sort("exon_start").with_columns(ranges = pl.struct("exon_start","exon_stop").map_elements(lambda x:[x["exon_start"],x["exon_stop"]])).group_by("transcript").agg(pl.col("ranges")).join(transcriptome.select('transcript',
 
 'strand',
 'chrom'), on="transcript").with_columns(local_range = pl.struct("ranges").map_elements(lambda x: list(np.cumsum([0] + [y[1]-y[0]+1 for y in x["ranges"]]))))

/tmp/ipykernel_194325/4278131432.py:4: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  ).filter(pl.col("transcript").is_in(set(samfile_mapped["transcript"]))).sort("exon_start").with_columns(ranges = pl.struct("exon_start","exon_stop").map_elements(lambda x:[x["exon_start"],x["exon_stop"]])).group_by("transcript").agg(pl.col("ranges")).join(transcriptome.select('transcript',
/tmp/ipykernel_194325/4278131432.py:7: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  'chrom'), on="transcript").with_columns(local_range = pl.struct("ranges").map_elements(lambda x: list(np.cumsum([0] + [y[1]-y[0]+1 for y in x["ranges"]]))))


In [55]:
# set(transcriptome["transcript"])-set(samfile_mapped["transcript"])

In [12]:
set(samfile_mapped["transcript"]) - set(exons["transcript"])

set()

In [9]:
samfile_mapped = samfile_mapped[samfile_mapped["seq"].apply(len)==36]

In [10]:
samfile_uniq = samfile_mapped.drop_duplicates(["qname"], keep=False)
samfile_duplicated = samfile_mapped[~samfile_mapped["qname"].isin(samfile_uniq["qname"])]#.groupby(list(samfile_mapped.columns[1:])).size().reset_index().rename(columns={0:'count'})


In [11]:
samfile_uniq.shape[0], samfile_duplicated.shape[0]

(82329, 3217216)

In [10]:
print("Uniq, Multimapped, MultiMapped After Deduplication")
samfile_uniq.shape[0], samfile_duplicated.shape[0], samfile_duplicated.drop_duplicates("qname").shape[0]

Uniq, Multimapped, MultiMapped After Deduplication


(120219, 5343730, 505929)

In [12]:
samfile_uniq_coding = samfile_uniq[samfile_uniq["transcript"].isin(transcriptome.filter(pl.col("tran_type") ==1 )["transcript"])]

In [13]:
print("Unique sequences in coding transcripts")
samfile_uniq_coding.shape

Unique sequences in coding transcripts


(68866, 6)

In [14]:
def genomic_pos(x):
    # print(x)
    idx = np.where(np.array(x["local_range"]) < x["pos"])[0][-1]
    genomic_range = x["ranges"][idx]
    if x["strand"] == "+":
        return genomic_range[0] + x["pos"] - x["local_range"][0]
    else:
        return genomic_range[1] - ( x["pos"] - x["local_range"][0])

In [15]:
samfile_duplicated = pl.from_pandas(samfile_duplicated).join(exons,on="transcript").with_columns(genomic_pos = pl.struct("pos","ranges","local_range","strand").map_elements(lambda x: genomic_pos(x) )).drop("ranges","local_range")
# .unique(["transcript","pos","chrom","genomic_pos"]).select("transcript","chrom","pos","genomic_pos")

/tmp/ipykernel_194325/1985116349.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  samfile_duplicated = pl.from_pandas(samfile_duplicated).join(exons,on="transcript").with_columns(genomic_pos = pl.struct("pos","ranges","local_range","strand").map_elements(lambda x: genomic_pos(x) )).drop("ranges","local_range")


In [16]:
samfile_duplicated_genomic_pos_counts = samfile_duplicated.group_by("qname").agg(pl.col("chrom").map_elements(lambda x: len(set(x))), pl.col("strand").map_elements(lambda x: len(set(x))), pl.col("genomic_pos").map_elements(lambda x: len(set(x))))#.unique(["seq","chrom","strand","gen"])

sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.


In [17]:
samfile_duplicated_genomic_pos_counts

qname,chrom,strand,genomic_pos
str,i64,i64,i64
"""SRR5882586.3501271_GCACC""",2,2,9
"""SRR5882586.5661951_CCACC""",1,1,8
"""SRR5882586.2204332_GTAGG""",1,1,3
"""SRR5882586.6394842_GTAGG""",1,1,6
"""SRR5882586.740695_AGGCA""",1,1,2
…,…,…,…
"""SRR5882586.6722340_GGCAC""",1,1,4
"""SRR5882586.6670406_GGCAC""",1,1,2
"""SRR5882586.861989_ACCAT""",1,1,9


In [18]:
343_893 + 82329

426222

In [19]:
samfile_duplicated_genomic_pos_counts_uniq = samfile_duplicated_genomic_pos_counts.filter((1 == pl.col("genomic_pos")) & (1 == pl.col("chrom")) & ( 1 == pl.col("strand")))

In [23]:
# Reads aligned to coding sequences, might aligned non coding even the genomic coordinates are the same
conding_duplicate_uniq = samfile_duplicated.filter(pl.col("qname").is_in(samfile_duplicated_genomic_pos_counts_uniq["qname"])).filter(pl.col("transcript").is_in(transcriptome.filter(pl.col("tran_type")==1)["transcript"]))#.select(samfile_uniq_coding.columns)#.drop("qname")
print("conding_duplicate_uniq")# .filter(~pl.col("qname").is_in(non_conding_duplicate_uniq["qname"]))
print(conding_duplicate_uniq.shape[0], conding_duplicate_uniq.unique("qname").shape)
# conding_duplicate_uniq = conding_duplicate_uniq.group_by(samfile_uniq_coding.columns[:-1]).count()

conding_duplicate_uniq
116546 (36796, 9)


In [76]:
conding_duplicate_uniq = conding_duplicate_uniq.group_by(samfile_uniq_coding.columns[:-1]).count()

/tmp/ipykernel_43643/1468714870.py:1: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  conding_duplicate_uniq = conding_duplicate_uniq.group_by(samfile_uniq_coding.columns[:-1]).count()


In [34]:
# None in set(transcriptome.filter(pl.col("tran_type")==1)["cds_start"])

In [27]:
conding_duplicate_uniq = conding_duplicate_uniq.drop(["strand","chrom","genomic_pos"])

In [28]:
conding_duplicate_uniq

qname,pos,seq,MD,NM,transcript
str,i64,str,str,str,str
"""SRR5882586.396_GGCAC""",248,"""CAGGCGAGGCTTCCCGCCTGGCGCATTACA…","""MD:Z:32A0A1C0""","""NM:i:3""","""ENST00000377777"""
"""SRR5882586.396_GGCAC""",248,"""CAGGCGAGGCTTCCCGCCTGGCGCATTACA…","""MD:Z:32A0A1C0""","""NM:i:3""","""ENST00000289316"""
"""SRR5882586.654_GGCAC""",191,"""CATGGCGGAGCCGTCGGCGGCCACTCAGTC…","""MD:Z:32A1T0C0""","""NM:i:3""","""ENST00000341307"""
"""SRR5882586.654_GGCAC""",187,"""CATGGCGGAGCCGTCGGCGGCCACTCAGTC…","""MD:Z:32A1T0C0""","""NM:i:3""","""ENST00000356000"""
"""SRR5882586.654_GGCAC""",179,"""CATGGCGGAGCCGTCGGCGGCCACTCAGTC…","""MD:Z:32A1T0C0""","""NM:i:3""","""ENST00000542238"""
…,…,…,…,…,…
"""SRR5882586.6956850_TAGGA""",372,"""CGGCCCGGGTGGGCCGGCGGCGGCCGCGGG…","""MD:Z:34A0G0""","""NM:i:2""","""ENST00000358154"""
"""SRR5882586.6956995_TAGGC""",352,"""GCCTCCGGGGGCTGCGTGCGCCCCGCGCGG…","""MD:Z:34A0A0""","""NM:i:2""","""ENST00000613122"""
"""SRR5882586.6956995_TAGGC""",1,"""GCCTCCGGGGGCTGCGTGCGCCCCGCGCGG…","""MD:Z:34A0A0""","""NM:i:2""","""ENST00000591598"""


In [29]:
all_uniq_36 = pl.concat([pl.from_pandas(samfile_uniq_coding),conding_duplicate_uniq])

In [32]:
all_uniq_36 = all_uniq_36.with_columns(seqlen=pl.col("seq").map_elements(len)).join(transcriptome.select("transcript","cds_start"),on="transcript")

sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredict

In [34]:
all_uniq_36_around_start = all_uniq_36.with_columns(cds_dist=pl.col("cds_start")- pl.col("pos")).filter((pl.col("cds_dist")>=10) & (pl.col("cds_dist")<=pl.col("seqlen")-10))

In [37]:
all_uniq_36_around_start.group_by("cds_dist").count().sort("count")

/tmp/ipykernel_194325/2714841418.py:1: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  all_uniq_36_around_start.group_by("cds_dist").count().sort("count")


cds_dist,count
i64,u32
15,81
16,83
26,120
18,154
17,167
…,…
12,298
13,408
10,480


In [39]:
# conding_duplicate_uniq_transcript_witj_longest_5utr = conding_duplicate_uniq.join(transcriptome.select("transcript","cds_start"),on="transcript").sort("cds_start", descending=True).unique(subset=["qname","transcript"], keep='first').group_by(["pos","seq","MD","NM","transcript"]).count()
conding_duplicate_uniq_transcript_witj_longest_5utr = conding_duplicate_uniq.join(transcriptome.select("transcript","cds_start"),on="transcript").group_by(["pos","seq","MD","NM","transcript"]).count()
#.sort("cds_start", descending=True).unique(subset=["qname","transcript"], keep='first')

/tmp/ipykernel_43643/89323400.py:2: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  conding_duplicate_uniq_transcript_witj_longest_5utr = conding_duplicate_uniq.join(transcriptome.select("transcript","cds_start"),on="transcript").group_by(["pos","seq","MD","NM","transcript"]).count()


In [79]:
samfile_coding_combined = pl.concat([pl.from_pandas(samfile_uniq_coding).with_columns(pl.col("count").cast(pl.UInt32)), conding_duplicate_uniq]).group_by(["pos","seq","MD","NM","transcript"]).sum()

In [80]:
samfile_coding_combined

pos,seq,MD,NM,transcript,count
i64,str,str,str,str,u32
3,"""AGCCGGCTTCCGGAAGCCGGGACGATGTCC…","""MD:Z:32A2A0""","""NM:i:2""","""ENST00000542946""",3
44,"""GCTCTGGCCGCGCCTGGCCTGGCCGGGACC…","""MD:Z:30A3G0C0""","""NM:i:3""","""ENST00000611653""",1
592,"""CCCCAGATTACCTGATGCAGCTGCTGTAG""","""MD:Z:23A2A1C0""","""NM:i:3""","""ENST00000361752""",1
2,"""GCGCCCGGGCGCTACTGGAAGAGGTCAAGG…","""MD:Z:33T0G0C0""","""NM:i:3""","""ENST00000636200""",2
312,"""CGGCAGCCGCACCTGCGCGGGCGACCAGCG…","""MD:Z:32A0G2""","""NM:i:2""","""ENST00000229795""",3
…,…,…,…,…,…
86,"""CCAGGACCACGGCTTCTTTCCTGCCAGATC…","""MD:Z:31A1C0C1""","""NM:i:3""","""ENST00000472520""",2
7,"""GAGCTCGCCGCGGCGGCGGCGGCGCTGCTG…","""MD:Z:27A0G1G2""","""NM:i:3""","""ENST00000268035""",2
931,"""TCATTCCCAACGGGGCCTTCGCGCACTGTA…","""MD:Z:26A1C0G1""","""NM:i:3""","""ENST00000232424""",2


In [41]:
# samfile_coding_combined

pos,seq,MD,NM,transcript,count
i64,str,str,str,str,u32
19,"""GCCGGGACATCCCGAGGAGCCGCGGTGAAA…","""MD:Z:31A2G0G0""","""NM:i:3""","""ENST00000236925""",2
1022,"""CCCCTGCCCTCTCTGAGGCAGGGGTGATGT…","""MD:Z:34A0G0""","""NM:i:2""","""ENST00000297338""",2
37,"""GGAGTTAATACACGAGGAACTCATGCACTG…","""MD:Z:27A0A1G2""","""NM:i:3""","""ENST00000518801""",1
161,"""GGCTCCGGGCCTCGCAGCCTCAGCCCCCGG…","""MD:Z:33A0G0C0""","""NM:i:3""","""ENST00000442588""",1
7768,"""GGACCCCCAGGCCATCAAGCCCATCCTGAA…","""MD:Z:33T0C0T0""","""NM:i:3""","""ENST00000300648""",2
…,…,…,…,…,…
119,"""GCCCAGCACGCCCCGGCCCCGCCCCAGCCC…","""MD:Z:33T0G0A0""","""NM:i:3""","""ENST00000263121""",1
2248,"""ACCTTCCCCACCCCCTCCTGGGAAGTGCCC…","""MD:Z:33C0A0C0""","""NM:i:3""","""ENST00000342665""",1
124,"""TGCTGGACGACACGGTGCCGCTGACAGCAG…","""MD:Z:32A1C0G0""","""NM:i:3""","""ENST00000383715""",2


In [57]:
samfile_coding_combined_with_trancrit_cds_start = samfile_coding_combined.join(transcriptome.select("transcript","cds_start"),on="transcript").with_columns(f_utr_dist = pl.col("cds_start") - pl.col("pos"), readlen = pl.col("seq").map_elements(len)).filter((pl.col("f_utr_dist")>=10) & (pl.col("f_utr_dist") <= (pl.col("readlen")-10)))

sys:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.


In [58]:
five_offsets = samfile_coding_combined_with_trancrit_cds_start.select("readlen","f_utr_dist","count").group_by(["readlen","f_utr_dist"]).sum().sort("readlen","count","f_utr_dist", descending=True)#.unique("readlen", keep="first")

In [60]:
five_offsets.sort(["readlen","count"], descending=True)

readlen,f_utr_dist,count
i64,i64,u32
36,21,488
36,11,483
36,10,480
36,13,408
36,12,298
…,…,…
29,19,16
29,17,16
29,14,3


In [61]:
five_offsets.unique("readlen", keep="first")

readlen,f_utr_dist,count
i64,i64,u32
36,21,488
33,18,47
32,20,58
31,15,39
30,20,20
29,11,24


In [62]:
five_offsets.filter(pl.col("readlen")==33)

readlen,f_utr_dist,count
i64,i64,u32
33,18,47
33,11,33
33,10,22
33,20,18
33,15,18
…,…,…
33,12,10
33,23,9
33,16,8


In [41]:
pl.from_pandas(samfile_uniq).join(transcriptome, on="transcript").with_columns(readlen = pl.col("seq").map_elements(len)).filter((pl.col("pos")< pl.col("cds_start")) & (pl.col("pos") + pl.col("readlen")-10 > pl.col("cds_start"))).with_columns(start_dist = pl.col("cds_start") - pl.col("pos")).select("readlen","start_dist").group_by("readlen").max()

/tmp/ipykernel_3732/3530999550.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  pl.from_pandas(samfile_uniq).join(transcriptome, on="transcript").with_columns(readlen = pl.col("seq").map_elements(len)).filter((pl.col("pos")< pl.col("cds_start")) & (pl.col("pos") + pl.col("readlen")-10 > pl.col("cds_start"))).with_columns(start_dist = pl.col("cds_start") - pl.col("pos")).select("readlen","start_dist").group_by("readlen").max()


readlen,start_dist
i64,i64
33,21
36,25
32,17
29,17
31,20
27,4
30,19


In [36]:
# samfile_duplicated_deduplicated

In [19]:
samfile_duplicated_deduplicated_gen_pos = pl.from_pandas(samfile_duplicated_deduplicated).join(gen_pos, on=["transcript","pos"])

In [27]:
samfile_duplicated_deduplicated_gen_pos

seq,pos,count,transcript,chrom,genomic_pos
str,i64,i64,str,str,i64
"""AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA…",81,4,"""ENST00000218089""","""chrX""",123960600
"""AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA…",82,4,"""ENST00000218089""","""chrX""",123960601
"""AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA…",83,4,"""ENST00000218089""","""chrX""",123960602
"""AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA…",84,4,"""ENST00000218089""","""chrX""",123960603
"""AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA…",85,4,"""ENST00000218089""","""chrX""",123960604
…,…,…,…,…,…
"""TTTTTGCCACATGTGGACAGCAAGTAGACC…",468,1,"""ENST00000618975""","""chr8""",103427565
"""TTTTTGGAATCAGCATTACAGGTGGCCTGT…",642,2,"""ENST00000169551""","""chr18""",74155787
"""TTTTTGGAATCAGCATTACAGGTGGCCTGT…",13,2,"""ENST00000579071""","""chr18""",74155188


In [30]:
samfile_duplicated_deduplicated_gen_pos.group_by(["chrom","genomic_pos"]).count().filter(pl.col("count") == 1).join(samfile_duplicated_deduplicated_gen_pos, on=["chrom","genomic_pos"], how="inner")["count_right"].sum()

/tmp/ipykernel_3732/1711134538.py:1: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  samfile_duplicated_deduplicated_gen_pos.group_by(["chrom","genomic_pos"]).count().filter(pl.col("count") == 1).join(samfile_duplicated_deduplicated_gen_pos, on=["chrom","genomic_pos"], how="inner")["count_right"].sum()


1937664

In [31]:
1937664 + 120219 # unambigiou

2057883

In [57]:
samfile_mapped_genome = pl.from_pandas(samfile_mapped[["qname","transcript","pos"]]).join(gen_pos,on=("transcript","pos"))

In [68]:
anm = samfile_mapped_genome.unique(["qname","chrom","genomic_pos"])

In [71]:
qname = anm.group_by("qname").count().filter(pl.col("count")>1)

/tmp/ipykernel_4325/513575387.py:1: DeprecationWarning: `GroupBy.count` is deprecated. It has been renamed to `len`.
  qname = anm.group_by("qname").count().filter(pl.col("count")>1)


In [77]:
# list(qname["qname"])

In [80]:
samfile_mapped[samfile_mapped["qname"].isin(list(qname["qname"]))]

,qname,rname,pos,seq,MD,NM,transcript
1,SRR5882586.2_TAGGC,ENST00000592665.1|ENSG00000130159.13|OTTHUMG00...,298,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000592665
2,SRR5882586.2_TAGGC,ENST00000590480.1|ENSG00000130159.13|OTTHUMG00...,501,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000590480
3,SRR5882586.2_TAGGC,ENST00000252440.11|ENSG00000130159.13|OTTHUMG0...,410,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000252440
4,SRR5882586.2_TAGGC,ENST00000270517.11|ENSG00000130159.13|OTTHUMG0...,447,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000270517
5,SRR5882586.2_TAGGC,ENST00000591104.5|ENSG00000130159.13|OTTHUMG00...,455,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000591104
...,...,...,...,...,...,...,...
11791288,SRR5882586.6957284_GGCAC,ENST00000599990.5|ENSG00000042753.11|OTTHUMG00...,131,TCCAGAACCGGGCAGGCAAGACGCGCCTGGCCCTGT,MD:Z:32A0A2,NM:i:2,ENST00000599990
11791289,SRR5882586.6957284_GGCAC,ENST00000597421.1|ENSG00000042753.11|OTTHUMG00...,79,TCCAGAACCGGGCAGGCAAGACGCGCCTGGCCCTGT,MD:Z:32A0A2,NM:i:2,ENST00000597421
11791290,SRR5882586.6957284_GGCAC,ENST00000263270.10|ENSG00000042753.11|OTTHUMG0...,246,TCCAGAACCGGGCAGGCAAGACGCGCCTGGCCCTGT,MD:Z:32A0A2,NM:i:2,ENST00000263270
11791291,SRR5882586.6957284_GGCAC,ENST00000352203.8|ENSG00000042753.11|OTTHUMG00...,31,TCCAGAACCGGGCAGGCAAGACGCGCCTGGCCCTGT,MD:Z:32A0A2,NM:i:2,ENST00000352203


In [10]:
dct["frequent_unmapped_reads"] = list(samfile_unmapped.sort_values("len",ascending=False).head(2000).itertuples(index=False,name=None))

In [29]:
test = samfile_mapped.groupby("qname").size().reset_index()

In [31]:
test[test[0]==1].shape

(120219, 2)

In [9]:
seq_len = samfile_unmapped["seq"].apply(len).values

In [10]:
max_seq_len = max(seq_len)

In [11]:
max_seq_len

36

In [12]:
# seq_len.values

In [13]:
data = []
for row in tqdm(pl.from_pandas(samfile_unmapped).iter_rows(named=True)):
    seq = row["seq"]
    seqlen = len(row["seq"])
    data.append(list(row["seq"])+[""]*(max_seq_len-seqlen)+[row["len"]])
nuc_dist = pd.DataFrame(data)
del data
nuc_dist.columns = nuc_dist.columns.map(str)
nuc_dist["length"] = seq_len

1212162it [00:04, 273312.23it/s]


In [64]:
# data[0]

In [14]:
nuc_count = []
for k in tqdm(range(max_seq_len)):
    nuc_dist["pos"] = k
    nuc_count.append(nuc_dist.groupby(["length", "pos",str(k)])[str(max_seq_len)].sum().reset_index().rename(columns={str(k):"nuc"}))
    # print(nuc_dist.groupby(["length", "pos",k])[max_seq_len].sum().reset_index().rename(columns={k:"nuc"}))
    # break
nuc_count = pd.concat(nuc_count)
    
# nuc_count = table2dict(pd.concat(nuc_count),["length","pos","nuc"])
    

100%|███████████████████████████████████████████████████████████████████████| 36/36 [00:04<00:00,  7.59it/s]


In [139]:
seq_len

31

In [16]:
def table2dict(table: pd.DataFrame, keys: list[str]) -> dict:
    '''
    Convert a table to a dictionary of lists. 
    >>> data = {'key1': [1, 2, 3], 'key2': [4, 5, 6], 'key3': [7, 8, 9], 
    ... 'key4': [10, 11, 12], 'key5': [13, 14, 15]}
    >>> table = pd.DataFrame(data)
    >>> table2dict(table, ['key1', 'key2', 'key3'])
    >>> {1:{4:{7:[10,13]}}, 2:{5:{8:[11,14]}}, 3:{6:{9:[12,15]}}}

    '''
    # print(keys)
    if not keys:
        return table.values.tolist()[0][0]
    key = keys[0]

    result = {}
    for k, group in table.groupby(key):
        if k == "":
            return {}
        res= table2dict(group.drop(columns=[key]), keys[1:])
        # print(k,res)
        if res:
            result[k] = res 
    return result

In [18]:
# table2dict(nuc_count,["length","pos","nuc"])

In [103]:
nuc_count[0].columns

Index(['length', 'pos', 'nuc', '36'], dtype='object')

In [84]:
pd.concat(nuc_count)

,length,pos,nuc,36
0,19,0,A,2
1,19,0,G,1
2,20,0,A,3
3,20,0,C,18
4,20,0,G,7
...,...,...,...,...
60,36,1,A,283285
61,36,1,C,2352784
62,36,1,G,1116004
63,36,1,N,454


In [89]:
nuc_dist#.groupby("length").size()

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,length,pos
0,A,A,A,A,A,A,A,A,G,G,...,T,A,G,,,,,1,32,0
1,A,A,A,A,A,A,A,G,T,T,...,A,G,,,,,,1,31,0
2,A,A,A,A,A,A,A,T,T,A,...,C,T,G,T,A,G,G,1,36,0
3,A,A,A,A,A,A,C,C,A,C,...,T,G,T,A,G,G,C,1,36,0
4,A,A,A,A,A,A,C,C,A,T,...,G,T,A,G,,,,1,33,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1212157,T,T,T,T,T,G,T,T,T,C,...,,,,,,,,1,29,0
1212158,T,T,T,T,T,T,C,A,C,T,...,T,G,T,A,G,G,C,3,36,0
1212159,T,T,T,T,T,T,C,C,C,A,...,G,T,T,C,A,C,A,1,36,0
1212160,T,T,T,T,T,T,G,G,G,C,...,,,,,,,,1,29,0


In [90]:
nuc_dist["pos"] = 1
table2dict(nuc_dist.groupby(["length", "pos",1])[36].sum().reset_index().rename(columns={1:"nuc"}),["length","pos","nuc"])

{19: {1: {'A': 1, 'C': 2}},
 20: {1: {'A': 11, 'C': 5, 'G': 13, 'T': 6}},
 21: {1: {'A': 71, 'C': 108, 'G': 75, 'T': 63}},
 22: {1: {'A': 203, 'C': 262, 'G': 199, 'T': 192}},
 23: {1: {'A': 201, 'C': 1181, 'G': 425, 'T': 348}},
 24: {1: {'A': 281, 'C': 857, 'G': 777, 'T': 259}},
 25: {1: {'A': 592, 'C': 645, 'G': 1549, 'T': 326}},
 26: {1: {'A': 1970, 'C': 704, 'G': 730, 'N': 1, 'T': 185}},
 27: {1: {'A': 1607, 'C': 727, 'G': 939, 'T': 309}},
 29: {1: {'A': 28664, 'C': 75163, 'G': 68140, 'N': 18, 'T': 37255}},
 30: {1: {'A': 43106, 'C': 127704, 'G': 86451, 'N': 37, 'T': 58837}},
 31: {1: {'A': 158844, 'C': 120675, 'G': 103195, 'N': 62, 'T': 123332}},
 32: {1: {'A': 50435, 'C': 173712, 'G': 218847, 'N': 54, 'T': 89157}},
 33: {1: {'A': 105337, 'C': 198555, 'G': 216814, 'N': 57, 'T': 89028}},
 36: {1: {'A': 283285, 'C': 2352784, 'G': 1116004, 'N': 454, 'T': 385546}}}

In [73]:
nuc_dist.groupby(["length", "pos",31])[36].sum().reset_index().rename(columns={31:"nuc"})

,length,pos,nuc,36
0,19,31,,3
1,20,31,,35
2,21,31,,317
3,22,31,,856
4,23,31,,2155
5,24,31,,2174
6,25,31,,3112
7,26,31,,3590
8,27,31,,3582
9,29,31,,209240


In [50]:
nuc_dist.groupby(0)[36].sum()

0
A     558962
C    1989204
G    1739818
N        330
T    2039062
Name: 36, dtype: int64

In [11]:
samfile_unmapped_nuc_count = nuc_count(pl.from_pandas(samfile_unmapped).select("seq","len"))

1159119it [07:21, 2625.23it/s]


KeyboardInterrupt: 

In [ ]:
dct["mapped_reads"]=samfile_mapped.drop_duplicates(["qname","seq"]).shape[0]

In [ ]:
dct["nuc_counts"] = {"mapped":table2dict(pd.DataFrame(samfile_mapped_nuc_count),[0,1,2]),"unmapped":table2dict(pd.DataFrame(samfile_unmapped_nuc_count),[0,1,2])}


In [9]:
# samfile_unmapped

In [10]:
pl.from_pandas(samfile_unmapped).select("seq","len")

seq,len
str,i64
"""AAAAAAAAGGCTCAGCCCACCAGTCGCTGT…",1
"""AAAAAAAGTTGGTGATGACATTGCCCTGTA…",1
"""AAAAAAATTAAATTTTAACCATGAGGGAAC…",1
"""AAAAAACCACCAGGTTCACTTCTTCCTACT…",1
"""AAAAAACCATATGCCAGGATTTTTGGACTG…",1
…,…
"""TTTTTGTTTCATGGCAACAAAACCTGTAG""",1
"""TTTTTTCACTGACCCGGTGAGGCGGGGGCT…",3
"""TTTTTTCCCAGTAAAAAAAAAAAAAAAGTG…",1


In [ ]:
# This is reverse of nuc_count, time consumig, improve algorithm here
dct["threeprime_nuc_count_dict"] =  {"mapped":table2dict(pd.DataFrame(samfile_mapped_nuc_count.with_columns(pl.col('pos') - pl.col('pos').max())),[0,1,2]),"unmapped":table2dict(pd.DataFrame(samfile_unmapped_nuc_count.with_columns(pl.col('pos') - pl.col('pos').max())),[0,1,2])}

In [ ]:
# samfile_unmapped_nuc_count = pl.concat(list(samfile_unmapped.apply(dinuc_count, axis=1))).group_by(["readlen","dinuc"]).sum()#.sort("readlen","pos","nuc")
samfile_mapped_dinuc_count = pl.concat(list(samfile_mapped.drop_duplicates(["qname","seq"]).groupby('seq').size().reset_index().rename(columns={0:'len'}).apply(dinuc_count, axis=1))).group_by(["readlen","dinuc"]).sum()#.sort("readlen","pos","nuc")

In [ ]:
dct["dinuc_counts"] = table2dict(pd.DataFrame(samfile_mapped_dinuc_count),[0,1])

In [20]:
samfile_mapped.shape

(5463949, 7)

In [24]:
# samfile_uniq = samfile_mapped.unique(subset=["qname","seq"], keep='none')
samfile_uniq = samfile_mapped.drop_duplicates(["qname"], keep=False)

In [15]:
samfile_uniq.shape

(120219, 7)

In [8]:
samfile_mapped.columns[1:]

Index(['rname', 'pos', 'seq', 'MD', 'NM', 'transcript'], dtype='object')

In [21]:
# samfile_duplicated = samfile_mapped.filter(~pl.col("qname").is_in(samfile_uniq["qname"])).drop("qname").group_by(samfile_mapped.columns[1:]).count()
# samfile_uniq = samfile_uniq.drop("qname").group_by(samfile_mapped.columns[1:]).count()
samfile_duplicated = samfile_mapped[~samfile_mapped["qname"].isin(samfile_uniq["qname"])].groupby(list(samfile_mapped.columns[1:])).size().reset_index().rename(columns={0:'count'})
samfile_uniq = samfile_uniq.groupby(list(samfile_mapped.columns[1:])).size().reset_index().rename(columns={0:'count'})
# del samfile_mapped
dct["ambiguous_counts"] = sum(samfile_duplicated["count"])

In [27]:
samfile_duplicated = samfile_mapped[~samfile_mapped["qname"].isin(samfile_uniq["qname"])]
samfile_duplicated.drop_duplicates("seq").shape

(174379, 7)

In [16]:
xx = pl.from_pandas(samfile_uniq).join(transcriptome.filter(pl.col("cds_start").is_not_null()).select("transcript", "cds_start"), on="transcript").with_columns(seqlen = pl.col("seq").str.len_chars(),dist = (pl.col("pos") - pl.col("cds_start")))

In [19]:
xx.filter((pl.col("dist") < 0) & (pl.col("seqlen") == 36))

qname,rname,pos,seq,MD,NM,transcript,cds_start,seqlen,dist
str,str,i64,str,str,str,str,i64,u32,i64
"""SRR5882586.546_AGGCA""","""ENST00000339950.4|ENSG00000162…",594,"""GTCACTCCCCCGCGGGGAGGGCGAGCCGAC…","""MD:Z:33A1T0""","""NM:i:2""","""ENST00000339950""",816,36,-222
"""SRR5882586.822_GCACC""","""ENST00000355703.3|ENSG00000197…",283,"""GCCAGGAGTGGGGACCCGGACCCCGCCCCT…","""MD:Z:31A2C1""","""NM:i:2""","""ENST00000355703""",540,36,-257
"""SRR5882586.1004_GTAGG""","""ENST00000229633.6|ENSG00000111…",77,"""GCCCAGCGTCAGGCGAGGGGCGACGTCTCG…","""MD:Z:35A0""","""NM:i:1""","""ENST00000229633""",198,36,-121
"""SRR5882586.1594_GGCAC""","""ENST00000303151.4|ENSG00000172…",199,"""GGCCTGGCGGGGAGCGGGCCTCGCGCGCCT…","""MD:Z:32A0G1G0""","""NM:i:3""","""ENST00000303151""",263,36,-64
"""SRR5882586.2345_GGCAC""","""ENST00000395270.5|ENSG00000196…",185,"""GTGAACCCCGGAGCCAGCGGCGCTGGGGCC…","""MD:Z:32A0G1G0""","""NM:i:3""","""ENST00000395270""",1042,36,-857
…,…,…,…,…,…,…,…,…,…
"""SRR5882586.6956290_GCACC""","""ENST00000267116.7|ENSG00000139…",63,"""CCGGTGGAGCCGCCGCCGCCGCCGCCGGGA…","""MD:Z:33C0G1""","""NM:i:2""","""ENST00000267116""",123,36,-60
"""SRR5882586.6956795_TGTAA""","""ENST00000328194.7|ENSG00000184…",117,"""ATGCGGTGCGGGGCGGCCCGGTGCCCCCTC…","""MD:Z:28C7""","""NM:i:1""","""ENST00000328194""",322,36,-205
"""SRR5882586.6956927_AGGCA""","""ENST00000402714.6|ENSG00000196…",488,"""CGGAGCCGCCGCGCCCGGGGCCGGCTCCCC…","""MD:Z:33T0A1""","""NM:i:2""","""ENST00000402714""",605,36,-117


In [62]:
xx= xx.filter((pl.col("dist").abs()>=10) & (pl.col("dist").abs() <= pl.col("seqlen")-10) )

In [43]:
# pd.DataFrame(xx.group_by("seqlen","dist").count().sort("seqlen","count")).values

In [58]:
transcript_pos = pl.from_pandas(samfile_duplicated[["transcript","pos"]].drop_duplicates())

In [36]:
transcriptome.filter((pl.col("tran_type")==1) & (pl.col("cds_start").is_null()))

transcript,cds_start,cds_stop,length,strand,chrom,tran_type
str,i64,i64,i64,str,str,i64


In [46]:
exons_order = exons.filter(pl.col("transcript").is_in(transctpit_pos["transcript"])).sort("exon_start").with_columns(ranges = pl.struct("exon_start","exon_stop").map_elements(lambda x:[x["exon_start"],x["exon_stop"]])).group_by("transcript").agg(pl.col("ranges")).join(transcriptome.select('transcript',
 
 'strand',
 'chrom'), on="transcript", how="inner").with_columns(ranges = pl.struct("ranges","strand").map_elements(lambda x: x["ranges"][::-1] if x["strand"] == "-" else  x["ranges"]) )

/tmp/ipykernel_43848/3035485516.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  exons_order = exons.filter(pl.col("transcript").is_in(transctpit_pos["transcript"])).sort("exon_start").with_columns(ranges = pl.struct("exon_start","exon_stop").map_elements(lambda x:[x["exon_start"],x["exon_stop"]])).group_by("transcript").agg(pl.col("ranges")).join(transcriptome.select('transcript',
/tmp/ipykernel_43848/3035485516.py:4: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  'chrom'), on="transcript", how="inner").with_columns(ranges = pl.struct("ranges","strand").map_elements(lambda x: x["ranges"][::-1] if x["strand"] == "-" else  x["ranges"]) )


In [49]:
exons_order.filter(pl.col("transcript")=="ENST00000517380")

transcript,ranges,strand,chrom
str,list[list[i64]],str,str
"""ENST00000517380""","[[133777300, 133777353], [133776971, 133777116], [133775552, 133776392]]","""-""","""chr8"""


In [34]:
exons_2 = exons.filter(pl.col("transcript").is_in(transctpit_pos["transcript"])).sort("exon_start").with_columns(ranges = pl.struct("exon_start","exon_stop").map_elements(lambda x:[x["exon_start"],x["exon_stop"]])).group_by("transcript").agg(pl.col("ranges")).join(transcriptome.select('transcript',
 
 'strand',
 'chrom'), on="transcript", how="inner").with_columns(local_range = pl.struct("ranges").map_elements(lambda x: list(np.cumsum([0] + [y[1]-y[0]+1 for y in x["ranges"]]))))

NameError: name 'transctpit_pos' is not defined

In [87]:
exons_2

transcript,ranges,strand,chrom,local_range
str,list[list[i64]],str,str,list[i64]
"""ENST00000372396""","[[43650158, 43650252], [43653137, 43653313], … [43704230, 43705515]]","""+""","""chr1""","[0, 95, … 4474]"
"""ENST00000624686""","[[206203888, 206208229]]","""+""","""chr1""","[0, 4342]"
"""ENST00000483189""","[[111241145, 111241336], [111243872, 111243966], … [111275532, 111276031]]","""+""","""chr13""","[0, 192, … 1205]"
"""ENST00000282032""","[[30323051, 30323202], [30330886, 30331374], … [30336567, 30338227]]","""+""","""chr11""","[0, 152, … 2430]"
"""ENST00000464438""","[[63520765, 63520894], [63521284, 63521526]]","""+""","""chr20""","[0, 130, 373]"
…,…,…,…,…
"""ENST00000427698""","[[232661695, 232661957], [232662802, 232662949], [232672309, 232672438]]","""+""","""chr2""","[0, 263, … 541]"
"""ENST00000244230""","[[71130314, 71130754], [71132898, 71133576], … [71149866, 71150101]]","""+""","""chr2""","[0, 441, … 2484]"
"""ENST00000517380""","[[133775552, 133776392], [133776971, 133777116], [133777300, 133777353]]","""-""","""chr8""","[0, 841, … 1041]"


In [96]:
def genomic_pos(x):
    # print(x)
    idx = np.where(np.array(x["local_range"]) < x["pos"])[0][-1]
    genomic_range = x["ranges"][idx]
    if x["strand"] == "+":
        return genomic_range[0] + x["pos"] - x["local_range"][0]
    else:
        return genomic_range[1] - ( x["pos"] - x["local_range"][0])

In [99]:
transcript_pos.join(exons_2,on="transcript").with_columns(genomic_pos = pl.struct("pos","ranges","local_range","strand").map_elements(lambda x: genomic_pos(x) )).unique(["transcript","chrom","genomic_pos"])

/tmp/ipykernel_43848/2029095953.py:1: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  transcript_pos.join(exons_2,on="transcript").with_columns(genomic_pos = pl.struct("pos","ranges","local_range","strand").map_elements(lambda x: genomic_pos(x) )).unique(["transcript","chrom","genomic_pos"])


transcript,pos,ranges,strand,chrom,local_range,genomic_pos
str,i64,list[list[i64]],str,str,list[i64],i64
"""ENST00000244221""",335,"[[71182740, 71188536], [71189846, 71190022], … [71226929, 71227084]]","""-""","""chr2""","[0, 5797, … 6279]",71188201
"""ENST00000420977""",354,"[[76699789, 76700538]]","""+""","""chr1""","[0, 750]",76700143
"""ENST00000309311""",1301,"[[3976057, 3976748], [3977216, 3977348], … [3985379, 3985470]]","""-""","""chr19""","[0, 692, … 3164]",3976872
"""ENST00000484403""",559,"[[29065061, 29065106], [29066717, 29067072], … [29073532, 29073616]]","""-""","""chr21""","[0, 46, … 808]",29068944
"""ENST00000310137""",1710,"[[66616582, 66617057], [66624214, 66625678], [66626461, 66627347]]","""+""","""chr11""","[0, 476, … 2828]",66625924
…,…,…,…,…,…,…
"""ENST00000284274""",2321,"[[14664748, 14664977], [14673642, 14673718], … [14692854, 14699711]]","""+""","""chr5""","[0, 230, … 7800]",14695175
"""ENST00000309594""",301,"[[21176645, 21176717], [21176912, 21179084]]","""+""","""chr13""","[0, 73, 2246]",21177213
"""ENST00000267116""",2156,"[[56237808, 56243393], [56243786, 56243877], … [56258244, 56258392]]","""-""","""chr12""","[0, 5586, … 8688]",56241237


In [18]:
non_coding = transcriptome.filter(pl.col("cds_start").is_null() | pl.col("cds_stop").is_null())

In [19]:
samfile_uniq["transcript"] = samfile_uniq.rname.apply(lambda x: x.split("|")[0].split(".")[0])

In [20]:
dct["noncoding_counts"] = samfile_uniq[samfile_uniq["transcript"].isin(non_coding["transcript"])]["count"].sum()

In [21]:
dct["coding_counts"] = samfile_uniq[~samfile_uniq["transcript"].isin(non_coding["transcript"])]["count"].sum()

In [22]:
dct["coding_counts"]

3870

In [10]:
samfile_uniq

,rname,pos,seq,MD,NM,transcript,count
0,ENST00000638157.1|ENSG00000023171.15|OTTHUMG00...,1436,TCCCCTTCACTGGACTTCAATGACACTGTAG,MD:Z:25A2A0G1,NM:i:3,ENST00000638157,1
1,ENST00000638157.1|ENSG00000023171.15|OTTHUMG00...,2431,CCAGTCTCAGACAGAATGGGCCCAGCCTGTAG,MD:Z:26T0C0T3,NM:i:3,ENST00000638157,2


In [14]:
# unambig_read_lengths
samfile_uniq["readlen"] = samfile_uniq.seq.apply(len)

In [34]:
dct["unambig_read_lengths"] = samfile_uniq.groupby("readlen")["count"].sum().to_dict()

In [15]:
#Use this to discard transcripts with no 5' leader or 3' trailer 
transcriptome_coding = transcriptome.filter((pl.col("cds_start") > 1) & (pl.col("cds_stop") <  pl.col("length")) & (pl.col("tran_type")==1)).select("transcript","cds_start","cds_stop")
# TODO: Might need to change start stop

In [27]:
# transcriptome.sort("cds_start").filter(~pl.col("cds_start").is_null())

In [16]:
xxx = pl.from_pandas(samfile_uniq[["pos","readlen","count","transcript"]]).join(transcriptome_coding, on="transcript", how="inner"
                                                                      ).with_columns(five_real_pos = pl.col("pos")-pl.col("cds_start"),
                                                                                     five_rel_stop_pos=pl.col("pos")-pl.col("cds_stop"),
                                                                                    three_real_pos=pl.col("pos")-pl.col("cds_start")+pl.col("readlen"),
                                                                                    three_rel_stop_pos=pl.col("pos")-pl.col("cds_stop")+pl.col("readlen")).with_columns(
five_frame = pl.col("five_real_pos") % 3, three_frame = pl.col("three_real_pos") % 3)

In [17]:
dct["fiveprime"] = table2dict(pd.DataFrame(xxx.filter((pl.col("five_real_pos") >= pl.col("cds_start")) & (pl.col("five_real_pos") <= pl.col("cds_stop"))).select("readlen","five_frame","count").group_by("readlen","five_frame").sum()),[0,1])
dct["threeprime"] = table2dict(pd.DataFrame(xxx.filter((pl.col("three_real_pos") >= pl.col("cds_start")) & (pl.col("three_real_pos") <= pl.col("cds_stop"))).select("readlen","five_frame","count").group_by("readlen","three_frame").sum()),[0,1])

NameError: name 'table2dict' is not defined

In [47]:
pd.DataFrame(xxx.select("readlen","five_frame","count").group_by("readlen","five_frame").sum()).columns

RangeIndex(start=0, stop=3, step=1)

In [83]:
# exons = pl.read_database(
#     query="SELECT * FROM exons", 
#     connection=conn,
# ).sort("transcript",'exon_start').group_by("transcript").agg(pl.col("exon_start"),pl.col("exon_stop"))#.sort("transcript")
# exons

In [67]:
xxz

readlen,count
i64,i64
32,5
36,4
31,1
33,1


In [18]:
# 5' offset
xxy = xxx.filter((pl.col("five_real_pos") > -600) & (pl.col("five_real_pos") < 600)).filter((pl.col("five_real_pos").abs() >=10) & (pl.col("five_real_pos").abs() <= (pl.col("readlen")-10)))
xxz=  xxy.select("readlen","count").group_by("readlen").max()
xxy.join(xxz, on=["readlen","count"], how="inner").select("readlen","five_real_pos").with_columns(pl.col("five_real_pos").abs()-2).group_by("readlen").max()


readlen,five_real_pos
i64,i64


In [19]:
# 3' offset
xxy = xxx.filter((pl.col("three_real_pos") > -600) & (pl.col("three_real_pos") < 600)).filter((pl.col("three_real_pos").abs() >=10) & (pl.col("three_real_pos").abs() <= (pl.col("readlen")-10)))
xxz=  xxy.select("readlen","count").group_by("readlen").max()
xxy.join(xxz, on=["readlen","count"], how="inner").select("readlen","three_real_pos").with_columns(pl.col("three_real_pos").abs()-2).group_by("readlen").max()


readlen,three_real_pos
i64,i64


In [78]:
sts = np.array(exons.filter(pl.col('transcript')=="INSB1")["exon_start"])[0]

In [80]:
stp = np.array(exons.filter(pl.col('transcript')=="INSB1")["exon_stop"])[0]

In [33]:
pl.DataFrame(samfile_mapped).join(transcriptome.select(""))

,qname,rname,pos,seq,MD,NM,transcript
1,SRR5882586.2_TAGGC,ENST00000592665.1|ENSG00000130159.13|OTTHUMG00...,298,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000592665
2,SRR5882586.2_TAGGC,ENST00000590480.1|ENSG00000130159.13|OTTHUMG00...,501,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000590480
3,SRR5882586.2_TAGGC,ENST00000252440.11|ENSG00000130159.13|OTTHUMG0...,410,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000252440
4,SRR5882586.2_TAGGC,ENST00000270517.11|ENSG00000130159.13|OTTHUMG0...,447,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000270517
5,SRR5882586.2_TAGGC,ENST00000591104.5|ENSG00000130159.13|OTTHUMG00...,455,GGGGCCACATTGACTTCATCTACCTGGCCCTGCGCT,MD:Z:35A0,NM:i:1,ENST00000591104
...,...,...,...,...,...,...,...
11791302,SRR5882586.6957294_CACCA,ENST00000604325.1|ENSG00000270935.1|OTTHUMG000...,101,GCTTTGCTTCTGATAATTTTGCAAACTGTAG,MD:Z:26A0C0G2,NM:i:3,ENST00000604325
11791303,SRR5882586.6957294_CACCA,ENST00000468359.1|ENSG00000144354.13|OTTHUMG00...,303,GCTTTGCTTCTGATAATTTTGCAAACTGTAG,MD:Z:26A0C0G2,NM:i:3,ENST00000468359
11791304,SRR5882586.6957294_CACCA,ENST00000467411.5|ENSG00000144354.13|OTTHUMG00...,190,GCTTTGCTTCTGATAATTTTGCAAACTGTAG,MD:Z:26A0C0G2,NM:i:3,ENST00000467411
11791305,SRR5882586.6957294_CACCA,ENST00000496441.5|ENSG00000144354.13|OTTHUMG00...,211,GCTTTGCTTCTGATAATTTTGCAAACTGTAG,MD:Z:26A0C0G2,NM:i:3,ENST00000496441
